In [39]:
import numpy as np
import matplotlib.pyplot as plt
import math
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import re
from typing import List, Dict

In [40]:
# this is used for new line splitter
def lang_chunk(document, chunk_size, chunk_overlap):

    #text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    text_splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    
    texts = text_splitter.split_text(document)

    
    return texts

In [77]:
def get_parts(text: str):
    text = text.replace("\\n", " ")

    text = re.sub(r"Conversation\s+\d+\s*", "", text)

    parts = re.split(r"(User:|Secretary:)", text)

    dialogues = []
    current_speaker = None

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part == "User:":
            current_speaker = "user"
        elif part == "Secretary:":
            current_speaker = "secretary"
        else:
            dialogues.append((current_speaker, " ".join(part.split())))

    return dialogues

In [106]:
def sliding_window_chunk(text: str, window_size: int, stride: int):
    parts = get_parts(text)
    window_size = 4
    stride = 1
    sliding_window = []
    for i in range(0, len(parts) - window_size + 1, stride):
        window = parts[i:i + window_size]

        chunk = "\n".join(f"{speaker}: {text}" for speaker, text in window)
        sliding_window.append(chunk)
    return sliding_window

In [107]:
def pair_dialogues(text: str):
    dialogues = get_parts(text)
    
    pairs = []
    i = 0
    while i < len(dialogues) - 1:
        if dialogues[i][0] == "user" and dialogues[i+1][0] == "secretary":
            pairs.append(
                f"user: {dialogues[i][1]}\nsecretary: {dialogues[i+1][1]}"
            )
            i += 2
        else:
            i += 1

    return pairs

In [108]:
with open("../data/secretary_conversations_dataset.txt", "r") as file:
    content = file.read()

converse = split_conversations(content)
i = 0

dialogues = []

for conv in converse:
    dialogues.extend(pair_dialogues(conv))

window = []

for conv in converse:
    window.extend(sliding_window_chunk(conv, 3, 1))

window


['user: Move my meeting with Emma.\nsecretary: Which meeting would you like to move?\nuser: The one at 1 PM Friday.\nsecretary: What time would you like to move it to?',
 'secretary: Which meeting would you like to move?\nuser: The one at 1 PM Friday.\nsecretary: What time would you like to move it to?\nuser: 10 AM.',
 'user: The one at 1 PM Friday.\nsecretary: What time would you like to move it to?\nuser: 10 AM.\nsecretary: Okay. I moved your meeting with Emma to 10 AM on Friday.',
 'user: Remind me to send the client report.\nsecretary: When would you like the reminder?\nuser: 4 PM.\nsecretary: Okay. I will remind you at 4 PM to send the client report.',
 'user: Schedule a meeting with Olivia.\nsecretary: Sure. What day would you like to schedule it?\nuser: Thursday.\nsecretary: What time works best?',
 'secretary: Sure. What day would you like to schedule it?\nuser: Thursday.\nsecretary: What time works best?\nuser: 2 PM.',
 'user: Thursday.\nsecretary: What time works best?\nuser: